# RAVDESS training allocation comparisons
Compare final-epoch predictions on identical test trials. Set ROOT and edit RUN_LABELS to select the experiment directories. Keep 17 as the baseline. Run all cells in order.

Run 17 (5 audio-only, 5 video-only, 10 audiovisual actors) is the baseline. Run 18 transfers three audio-only actors to video-only. Run 19 transfers three video-only actors to audio-only. Run 20 transfers three actors from each unimodal condition to audiovisual. Interpret these as allocation contrasts. Each contrast can also reflect which actors were assigned.

For each test condition, compute one row-normalized confusion matrix per fold/block. Pair matching fold/block IDs across runs. A diagonal cell is recall (accuracy among trials of that true emotion). Off-diagonal cells give the fraction assigned to each alternative. Differences are comparison minus baseline in percentage points. Two-sided paired t tests use fold/block models as the unit, with df = models − 1. Blocks share test actors within a fold, and training sets overlap across folds. Model-level t-test p values are exploratory because these differences are dependent. P values are unadjusted across the many cell tests. Error bars show sample standard deviations across models. Constant paired differences have zero estimated variance and are reported as undefined.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from IPython.display import display

ROOT = Path(r'C:/Users/sosc/Desktop/alex/visual_audio_emotion/HiCMAE_results')
RUN_LABELS = {17:'a5v5av10', 18:'a2v8av10', 19:'a8v2av10', 20:'a2v2av16'}
if 17 not in RUN_LABELS:
    raise ValueError('Include baseline Run 17 in RUN_LABELS.')
comparisons = [r for r in RUN_LABELS if r != 17]
RUNS = {r: ROOT / f'run{r}' for r in RUN_LABELS}
def offsets(n):
    return (np.arange(n) - (n-1)/2) * .65 / max(n,1)
OUT = ROOT / 'allocation_comparisons'
OUT.mkdir(exist_ok=True, parents=True)

conditions = ['audio_neutral','video_noise','both']
emotions = ['happy','sad','angry','fearful','surprised','disgusted']
keys = ['fold','block','condition','actor','trial']

In [ ]:
def load_run(run, path):
    cfg = json.loads((path/'experiment.json').read_text())
    if 'a{}v{}av{}'.format(*cfg['train_actors']) != RUN_LABELS[run] or cfg['emotions'] != emotions:
        raise ValueError(f'Check allocation or class order for Run {run}')
    blocks = []
    for fold in cfg['holdout_groups']:
        for block in range(1, cfg.get('blocks_per_fold', 1)+1):
            folder = path / f'fold_{fold:02d}'
            if 'blocks_per_fold' in cfg:
                folder = folder / f'block_{block:02d}'
            folder = folder / f"epoch_{cfg['epochs']:03d}"
            for cond in conditions:
                d = pd.read_csv(folder/f'test_{cond}.csv')
                d['trial'] = d.video.str.replace(chr(92),'/',regex=False).str.split('/').str[-1]
                if not d.condition.eq(cond).all() or not d[['label','prediction']].isin(range(6)).all().all():
                    raise ValueError(f'Invalid labels or condition in {folder}')
                blocks.append(d.assign(fold=fold, block=block))
    trials = pd.concat(blocks).set_index(keys, verify_integrity=True).sort_index()
    return cfg, trials

runs = {r: load_run(r,p) for r,p in RUNS.items()}
cfg0, base = runs[17]
for r,(cfg,d) in runs.items():
    if not d.index.equals(base.index) or not d.label.equals(base.label):
        raise ValueError(f'Unmatched trials, actors, labels or folds in Run {r}')
    for k in ['classification','epochs','mlp_hidden_dims','seed','lr','batch_size']:
        if cfg[k] != cfg0[k]:
            print(f'Additional difference: Run {r}, {k}: {cfg[k]} vs {cfg0[k]}')
print('Matched test presentations per run:', len(base))

In [ ]:
rows = []
for r, (_, d) in runs.items():
    for (fold,block,c), g in d.groupby(level=['fold','block','condition']):
        counts = pd.crosstab(g.label,g.prediction).reindex(index=range(6), columns=range(6), fill_value=0)
        totals = counts.sum(axis=1)
        if (totals==0).any(): raise ValueError('Missing true emotion in fold')
        for i,true in enumerate(emotions):
            for j,pred in enumerate(emotions):
                rows.append([r,fold,block,c,true,pred,counts.iloc[i,j],totals.iloc[i],100*counts.iloc[i,j]/totals.iloc[i]])
folds = pd.DataFrame(rows,columns=['run','fold','block','condition','true','pred','count','total','percent'])
folds.to_csv(OUT/'fold_confusion_cells.csv',index=False)
wide = folds.pivot(index=['condition','true','pred','fold','block'],columns='run',values='percent')
rows = []
for (c,true,pred), g in wide.groupby(level=['condition','true','pred']):
    for r in comparisons:
        delta = g[r]-g[17]
        valid = len(delta)>1 and delta.std()>1e-12
        t,p = ttest_rel(g[r],g[17]) if valid else (np.nan,np.nan)
        rows.append([r,c,true,pred,len(delta),g[17].mean(),g[r].mean(),delta.mean(),delta.std(ddof=1),t,len(delta)-1,p])
results = pd.DataFrame(rows,columns=['run','condition','true','pred','n','baseline_pct','comparison_pct','delta_pp','std_pp','t','df','p'])
results['allocation'] = results.run.map(RUN_LABELS)
accuracy = results.query('true == pred').copy()
results.to_csv(OUT/'paired_confusion_tests.csv',index=False)
accuracy.to_csv(OUT/'paired_emotion_accuracy.csv',index=False)
with pd.option_context('display.max_rows',None):
    display(accuracy.drop(columns='pred'))

In [ ]:
diagonal = folds[folds.true == folds.pred]
performance = diagonal.groupby(['run','condition','true']).percent.agg(['mean','std'])
fig, axes = plt.subplots(1, 3, figsize=(17,5), sharey=True, constrained_layout=True)
for ax, condition in zip(axes, conditions):
    for offset, run in zip(offsets(len(RUN_LABELS)), RUN_LABELS):
        g = performance.loc[(run, condition)].reindex(emotions)
        ax.errorbar(np.arange(len(emotions))+offset, g['mean'], yerr=g['std'],
                    fmt='o', capsize=3, label=RUN_LABELS[run])
    ax.set(xticks=np.arange(len(emotions)), xticklabels=emotions, title=condition)
    ax.tick_params(axis='x', rotation=45)
axes[0].set_ylabel('Per-emotion accuracy (%, model mean ± SD)')
axes[-1].legend(title='Training allocation')
fig.savefig(OUT/'emotion_accuracy_all_runs.png', dpi=180)
plt.show()

## Audiovisual accuracy gain by emotion
Each point shows the mean within-model gain for one training allocation. Positive values indicate higher audiovisual accuracy. Error bars show ±1 SD across fold/block model differences.

In [ ]:
paired = diagonal.pivot(index=['run', 'fold', 'block', 'true'], columns='condition', values='percent')
if paired[conditions].isna().any().any():
    raise ValueError('Missing condition for a run, fold or emotion.')
for control, label in [('audio_neutral', 'audio only'), ('video_noise', 'video only')]:
    gain = (paired['both'] - paired[control]).groupby(level=['run', 'true']).agg(['mean', 'std'])
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    for offset, (run, allocation) in zip(offsets(len(RUN_LABELS)), RUN_LABELS.items()):
        points = gain.loc[run].reindex(emotions)
        ax.errorbar(np.arange(len(emotions)) + offset, points['mean'], yerr=points['std'],
                    fmt='o', capsize=3, label=allocation)
    ax.axhline(0, color='gray', linewidth=1)
    ax.set(xticks=np.arange(len(emotions)), xticklabels=emotions, xlabel='Emotion',
           ylabel='Accuracy difference (percentage points)', title=f'Audiovisual minus {label}')
    ax.legend(title='Training allocation')
    fig.savefig(OUT / f'audiovisual_minus_{label.replace(" ", "_")}.png', dpi=180)
    plt.show()

In [ ]:
for r in comparisons:
    fig, axes = plt.subplots(1,3,figsize=(18,6),constrained_layout=True)
    sub = results[results.run==r]
    limit = max(1, sub.delta_pp.abs().max())
    for ax,c in zip(axes,conditions):
        cells = sub[sub.condition==c].set_index(['true','pred'])
        grid = cells.delta_pp.unstack().loc[emotions,emotions]
        im=ax.imshow(grid,cmap='RdBu_r',vmin=-limit,vmax=limit)
        for i,e in enumerate(emotions):
            for j,q in enumerate(emotions):
                v,p = cells.loc[(e,q),['delta_pp','p']]
                label = f'p={p:.3g}' if np.isfinite(p) else 'p=n/a'
                ax.text(j,i,f'{v:+.1f}\n{label}',ha='center',va='center',fontsize=8,
                        color='white' if abs(v)>.6*limit else 'black')
        ax.set(xticks=range(6),yticks=range(6),xticklabels=emotions,yticklabels=emotions,
               xlabel='Predicted emotion',ylabel='True emotion',title=c)
        ax.tick_params(axis='x',rotation=45)
    fig.colorbar(im,ax=axes,label='Difference (percentage points)',shrink=.7)
    fig.suptitle(f'{RUN_LABELS[r]} minus {RUN_LABELS[17]}')
    fig.savefig(OUT/f'confusion_run{r}_minus17.png',dpi=180)
    plt.show()

fig,axes=plt.subplots(1,3,figsize=(17,5),sharey=True,constrained_layout=True)
for ax,c in zip(axes,conditions):
    for offset,r in zip(offsets(len(comparisons)),comparisons):
        g=accuracy[(accuracy.condition==c)&(accuracy.run==r)].set_index('true').loc[emotions]
        x = np.arange(len(emotions)) + offset
        line = ax.errorbar(x, g.delta_pp, yerr=g.std_pp, fmt='o', capsize=3,
                           label=f'{RUN_LABELS[r]} − {RUN_LABELS[17]}')
        significant = g.p.lt(.05).to_numpy()
        ax.scatter(x[significant], g.delta_pp.to_numpy()[significant], marker='*',
                   s=160, color=line.lines[0].get_color(), edgecolors='black',
                   linewidths=.8, zorder=5)
    ax.axhline(0,color='gray',lw=1)
    ax.set(xticks=range(6),xticklabels=emotions,title=c)
    ax.tick_params(axis='x',rotation=45)
fig.suptitle('★  Paired difference from baseline (exploratory): two-sided p < 0.05 (unadjusted)')
axes[0].set_ylabel('Accuracy difference (percentage points, mean ± paired SD)')
axes[-1].legend()
fig.savefig(OUT/'emotion_accuracy_differences.png',dpi=180)
plt.show()